In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model import NSConfig, NsModel, CUR_DIR

In [3]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f"{CUR_DIR}/model/tokenizer_fixed.json")

In [4]:
cfg = NSConfig()

device = "cuda:1"

model = NsModel(cfg)
model.to(device)

model.load_state_dict(
    torch.load(f"{CUR_DIR}/model/ns_model_v2_epoch_1_step_10w.bin", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModel(
  (emb): Embedding(32000, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32000, bias=False)
)

In [8]:
prompt = "the cat jump over the dog"

input_ids = tokenizer.encode(prompt).ids

eos_token_id = tokenizer.token_to_id("<eos>")
input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

In [ ]:
# no kv-cache

max_output = 200

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_ids)

        logits = logits[:, -1, :]

        # 贪心
        # next_token = torch.argmax(logits, dim=-1, keepdim=True)

        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break
        
        input_ids = torch.cat([input_ids, next_token], dim=1)




output = tokenizer.decode(input_ids.squeeze(0).cpu().numpy())
print(output)


 the cat jump over the dog’s head, Chelsea: “It takes us a day to not be calm. [describing one of] every four months [Episode 121]. In one go, later, when I would frustrated my brain, I would pitch the cat “So says the Frederick that working as a tree makes the most sense.” So the dogs would make their best friends, with the city in it.
The fifth proper holiday for the cat. It was among a class five cleanser set out at Merryland in rural New York. It was:
We must have much of the countryside: lots of bungalows outside and dog-sitting near other frees does not help either. Sociology: there are areas I’d like to get help from in suburbanston but can spoil the rest of the lab.
A young young man has built buildings in an area that contain about half of his mechanical equipment. It might be an eighteen foot TV in use since you can make a certain amount


In [ ]:
# enable kvcache

max_output = 1500

with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_ids, use_cache=True)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break

        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True)

    
output = tokenizer.decode(generated.squeeze(0).cpu().numpy())
print(output)


 the cat jump over the dog himself.'"
Last week, on the eve of the Twie Pops fame killings, The Christian Pops set out to kill Eric Owens in the wake of Rylander's supporting testimony. Gore was waiting to hear what he said of the subject. Despite his statement, Owens says judges actually don't c'est recolort, even if they are merely sawers.
Yesterday, given the evidence we allow the newspaper's "brainless" awarding Vicki Rucker, who is still matching Christie's brilliant corporate takeover angle, we turned our attention to Maria Mertz, a goat lawyer from Seattle. Her repast 53 shows the impatient Cowart impersonator(who has confessed to beat Mertz’s "newsy" husband on Twitter) with some of the bristling, sweet-girl-yah-girl wrenching comments.
I don’t particularly like Kaley Cuoco’s description about Second Man, which includes Amelie Long, reenacting the Barbara Woltz trials. (For these reasons, I am reviewing this.)
Can they debate whether 13-year-old Witlda Mezzetta is in her 20th b